# Módulo 2 · Clase 3 — GitHub y repositorios remotos

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Bloque | Qué aprenderás |
|---|--------|----------------|
| 1 | Git vs GitHub | La diferencia fundamental |
| 2 | Crear cuenta y primer repo en GitHub | Configuración inicial de la plataforma |
| 3 | SSH vs HTTPS | Cómo autenticarte de forma segura |
| 4 | `git remote` | Conectar tu repo local con el remoto |
| 5 | `git push` | Subir cambios al remoto |
| 6 | `git clone` | Descargar un repo existente |
| 7 | `git fetch` y `git pull` | Recibir cambios del remoto |
| 8 | Fork | Contribuir a proyectos ajenos |
| 9 | Práctica | Publicar el sistema de notas en GitHub |

> 💡 **Idea central:** Git es la herramienta local. GitHub es la plataforma donde el trabajo se comparte, revisa y colabora. Hoy aprendemos a conectar los dos mundos.

---
## 1. Git vs GitHub — No son lo mismo

```
GIT                                GITHUB
───────────────────────────────    ──────────────────────────────────
Herramienta de línea de comandos   Plataforma web (empresa de Microsoft)
Corre en tu máquina                Corre en servidores de GitHub
Gestiona el historial local        Aloja repositorios remotos
Creado por Linus Torvalds (2005)   Fundado en 2008
Open source                        Servicio freemium
Sin él, GitHub no funciona         Sin él, Git sigue funcionando solo
```

**Alternativas a GitHub** (todas usan Git):

| Plataforma | Cuándo se usa |
|------------|---------------|
| **GitLab** | Empresas que quieren self-hosting o CI/CD integrado |
| **Bitbucket** | Equipos que usan el ecosistema Atlassian (Jira, Confluence) |
| **Azure DevOps** | Entornos Microsoft / .NET |
| **Gitea** | Self-hosted, muy ligero |

---
## 2. Crear cuenta y primer repositorio en GitHub

### Paso a paso (hacer en clase)

1. Ve a **github.com** → Sign up
2. Elige un nombre de usuario (será parte de tus URLs: `github.com/tu-usuario`)
3. Confirma el email
4. Crear un repositorio nuevo:
   - **Repository name:** `sistema-notas`
   - **Description:** Sistema de notas en línea de comandos
   - **Visibility:** Public (para que todos puedan verlo)
   - ⬛ **No** inicialices con README — ya tenemos uno local
   - Clic en **Create repository**

### Qué ves al crear un repo vacío

GitHub te muestra exactamente los comandos para conectar tu repo local. Es la página más útil de toda la plataforma:

```bash
# Si ya tienes el repo local (nuestro caso):
git remote add origin https://github.com/TU-USUARIO/sistema-notas.git
git branch -M main
git push -u origin main
```

---
## 3. SSH vs HTTPS — Cómo autenticarte

Para que Git pueda subir cambios a GitHub, tiene que identificarte. Hay dos métodos:

```
HTTPS                              SSH
─────────────────────────────────  ──────────────────────────────────
https://github.com/user/repo.git   git@github.com:user/repo.git
Te pide usuario/token cada vez*    Autenticación automática con clave
Más fácil de configurar            Más seguro y cómodo a largo plazo
Funciona en redes restrictivas     Puede bloquearse en algunas redes

* Con credential manager se guarda.
```

**Recomendación:** configura SSH — lo haces una vez y olvidas las contraseñas para siempre.

### Configurar SSH en GitHub (una sola vez)

In [ ]:
# 1. Verificar si ya tienes claves SSH
!ls ~/.ssh/*.pub 2>/dev/null && echo "Ya tienes claves SSH" || echo "No tienes claves SSH aún"

In [ ]:
# 2. Generar una clave SSH nueva (solo si no tienes)
# Sustituye el email por el de tu cuenta de GitHub
#
# !ssh-keygen -t ed25519 -C "tu@email.com" -f ~/.ssh/id_ed25519 -N ""
#
# ed25519 es el algoritmo moderno recomendado
# -N "" crea la clave sin passphrase (más cómodo para el día a día)

print("Para generar la clave SSH, ejecuta en el terminal:")
print('ssh-keygen -t ed25519 -C "tu@email.com"')
print()
print("Luego muestra la clave pública con:")
!cat ~/.ssh/id_ed25519.pub 2>/dev/null || echo "(genera la clave primero)"

### Añadir la clave pública a GitHub

1. Copia el contenido de `~/.ssh/id_ed25519.pub` (toda la línea)
2. Ve a **GitHub → Settings → SSH and GPG keys → New SSH key**
3. Pega la clave y dale un nombre descriptivo (ej: `Portátil MacBook`)
4. Clic en **Add SSH key**

In [ ]:
# Verificar que la conexión SSH con GitHub funciona
!ssh -T git@github.com 2>&1 || true
# Respuesta esperada: "Hi TU-USUARIO! You've successfully authenticated..."

---
## 4. `git remote` — Conectar local con remoto

Un **remote** es un alias que apunta a una URL de repositorio. El nombre convencional para el remoto principal es **`origin`**.

```bash
git remote add <nombre> <url>    # añadir un remoto
git remote -v                    # ver remotos configurados
git remote remove <nombre>       # eliminar un remoto
git remote rename <old> <new>    # renombrar
git remote set-url <nombre> <url> # cambiar la URL
```

In [ ]:
import os
from pathlib import Path

# Situarnos en el repo sistema_notas de la clase anterior
NOTAS = Path.home() / "sistema_notas"
if NOTAS.exists():
    os.chdir(NOTAS)
    print(f"Repo: {Path.cwd()}")
    !git log --oneline
else:
    print("⚠️  Crea primero el repo sistema_notas (Clase 2, Ejercicio 1)")

In [ ]:
# Conectar con GitHub
# CAMBIA esta URL por la de tu repositorio en GitHub
GITHUB_URL = "git@github.com:TU-USUARIO/sistema-notas.git"   # SSH
# GITHUB_URL = "https://github.com/TU-USUARIO/sistema-notas.git"  # HTTPS

# !git remote add origin {GITHUB_URL}

# Ver los remotos configurados
!git remote -v

---
## 5. `git push` — Subir cambios al remoto

```bash
git push -u origin main    # primer push: establece el tracking
git push                   # pushes siguientes (ya sabe a dónde)
git push origin main       # push explícito de una rama
git push --tags            # también subir los tags
```

**¿Qué hace `-u`?**  
`-u` (o `--set-upstream`) le dice a Git que la rama local `main` hace tracking de la rama remota `origin/main`. Después del primer push con `-u`, ya puedes usar solo `git push` y `git pull`.

```
LOCAL                              REMOTO (GitHub)
──────                             ──────────────────
main ────── git push ──────────►  origin/main
main ◄───── git pull ──────────── origin/main
```

In [ ]:
# Asegurarnos de estar en main
!git branch -M main

# Primer push (descomentar cuando tengas el remote configurado)
# !git push -u origin main

# Verificar el estado de tracking
!git branch -vv

In [ ]:
# Subir también los tags (los tags no se suben con git push normal)
# !git push --tags

# O subir un tag concreto
# !git push origin v1.0

print("Para subir los tags al remoto:")
print("  git push --tags         # todos los tags")
print("  git push origin v1.0   # un tag concreto")

---
## 6. `git clone` — Descargar un repo existente

`git clone` descarga un repositorio completo — con todo su historial — y configura automáticamente el remote `origin`.

```bash
git clone <url>                  # clona en carpeta con nombre del repo
git clone <url> nombre-local     # clona con nombre de carpeta personalizado
git clone <url> --depth 1        # shallow clone: solo el último commit (más rápido)
```

**Diferencia entre `clone` e `init + remote add`:**

| | `git clone` | `git init` + `git remote add` |
|-|-------------|--------------------------------|
| Cuándo | El repo YA existe en GitHub | Tienes código local que quieres subir |
| Qué hace | Descarga todo + configura origin | Creas el repo desde cero |
| Historial | Lo descarga completo | Empieza vacío |

In [ ]:
import os, shutil
from pathlib import Path

# Clonar un repo público real (sin necesitar autenticación)
DESTINO = Path.home() / "clone_demo"
if DESTINO.exists():
    shutil.rmtree(DESTINO)

os.chdir(Path.home())

# Clonamos un repo pequeño y público como ejemplo
!git clone https://github.com/octocat/Hello-World clone_demo

os.chdir(DESTINO)
print("\nContenido del repo clonado:")
!ls -la
print("\nHistorial:")
!git log --oneline
print("\nRemote configurado automáticamente:")
!git remote -v

---
## 7. `git fetch` y `git pull` — Recibir cambios

Cuando alguien (o tú desde otro ordenador) hace un push al remoto, tienes que traer esos cambios a tu máquina.

```
git fetch                         git pull
──────────────────────────────    ──────────────────────────────────
Descarga cambios del remoto       Descarga Y fusiona en tu rama actual
NO toca tu working directory      SÍ modifica tu working directory
Seguro, siempre puedes hacerlo    Puede crear conflictos
Tienes que hacer merge después    fetch + merge en un solo paso

git pull = git fetch + git merge
```

**¿Cuál usar?**
- `git fetch` → cuando quieres ver qué hay en el remoto antes de integrar.
- `git pull` → cuando confías en que no habrá conflictos (trabajo en solitario).
- En equipos: `git fetch` + revisar + `git merge` o `git rebase`.

In [ ]:
os.chdir(DESTINO)

# fetch descarga la información sin tocar tus archivos
!git fetch origin
print("\nTras git fetch:")
!git log --oneline --all   # --all muestra también las ramas remotas

# Ver diferencia entre local y remoto
print("\nDiferencias entre local y origin/main:")
!git log HEAD..origin/main --oneline 2>/dev/null || echo "Estás al día"

In [ ]:
# pull = fetch + merge
# (En este repo de demo no habrá cambios nuevos, pero el comando es el mismo)
!git pull origin main

print("\nEstado tras pull:")
!git status

### El flujo cuando el remoto tiene cambios nuevos

```
SITUACIÓN HABITUAL EN UN EQUIPO:

Remoto:  A ── B ── C ── D   (tu compañero hizo C y D)
Local:   A ── B ── E         (tú hiciste E sin saber de C y D)

git fetch:    Descarga C y D, los conoce pero no los fusiona
git merge:    A ── B ── C ── D ── E ── M  (M = merge commit)
git rebase:   A ── B ── C ── D ── E'      (historial lineal, más limpio)

git pull --rebase  →  hace fetch + rebase automáticamente
```

---
## 8. Fork — Contribuir a proyectos ajenos

Un **fork** es una copia personal de un repositorio ajeno en tu cuenta de GitHub. Te permite proponer cambios a proyectos donde no tienes permisos de escritura.

```
FLUJO DE CONTRIBUCIÓN CON FORK:

1. Fork del repo original ──────►  tu-usuario/proyecto (en GitHub)
                                          │
2. git clone tu fork ─────────────────────►  tu máquina
                                          │
3. Crear rama: git switch -c mi-mejora    │
                                          │
4. Hacer cambios + commits                │
                                          │
5. git push origin mi-mejora ─────────────►  tu fork en GitHub
                                          │
6. Abrir Pull Request ──────────────────► proyecto-original/main
                                          │
7. El maintainer revisa y acepta (merge)  │
```

**Fork vs Clone:**

| | Clone | Fork |
|-|-------|------|
| Dónde | Solo en tu máquina | En GitHub + tu máquina |
| Puedes hacer push | Solo si tienes permisos | Siempre (es tuyo) |
| Para contribuir | No | Sí |

In [ ]:
# Cuando tienes un fork, es buena práctica mantener sincronizado
# el remoto del proyecto original (llamado 'upstream' por convención)

print("Flujo para mantener un fork sincronizado:")
flujo = """
# Añadir el repo original como 'upstream'
git remote add upstream https://github.com/ORIGINAL/proyecto.git

# Ver todos los remotos
git remote -v
# origin    git@github.com:TU-USUARIO/proyecto.git  (tu fork)
# upstream  https://github.com/ORIGINAL/proyecto.git (el original)

# Sincronizar tu fork con el original
git fetch upstream
git merge upstream/main
git push origin main
"""
print(flujo)

---
## 9. Práctica — Publicar el sistema de notas en GitHub

En esta práctica publicamos el repo `sistema_notas` de la Clase 2 en GitHub y practicamos el flujo completo push/pull.

> ⚠️ Para los ejercicios 2–5 necesitas tener configurado un remote en GitHub. El ejercicio 1 funciona sin cuenta.

---

### Ejercicio 1 — Preparar el repo para publicar

**Enunciado:** Antes de subir a GitHub, un repo profesional necesita ciertos archivos. Añádelos al repo `sistema_notas`:

1. **`README.md` completo** con: título, descripción, sección de instalación, sección de uso y sección de estructura del proyecto.
2. **`LICENSE`** con la licencia MIT (búscala en choosealicense.com y pégala).
3. **`requirements.txt`** con las dependencias del proyecto (en este caso ninguna, pero el archivo debe existir vacío o con un comentario).
4. Commitea los tres archivos con el mensaje apropiado.
5. Verifica el historial final.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import os
from pathlib import Path

NOTAS = Path.home() / "sistema_notas"
os.chdir(NOTAS)
print(f"Trabajando en: {Path.cwd()}")


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
from datetime import date

# README.md completo
Path("README.md").write_text(f"""# Sistema de Notas

Aplicación de línea de comandos para gestionar notas personales.
Proyecto de práctica del Curso de IA.

## Instalación

```bash
git clone https://github.com/TU-USUARIO/sistema-notas.git
cd sistema-notas
```

## Uso

```python
from gestor import GestorNotas

g = GestorNotas()
g.crear('Reunión', 'Preparar presentación del lunes')
g.listar()
resultados = g.buscar('reunión')
g.eliminar(1)
```

## Estructura

```
sistema_notas/
├── nota.py          # Modelo de datos
├── gestor.py        # Lógica de negocio
├── test_gestor.py   # Pruebas
└── README.md        # Este archivo
```

## Licencia

MIT © {date.today().year}
""", encoding="utf-8")

# LICENSE — MIT
Path("LICENSE").write_text(f"""MIT License

Copyright (c) {date.today().year} Curso IA

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
""", encoding="utf-8")

# requirements.txt
Path("requirements.txt").write_text(
    "# Este proyecto no tiene dependencias externas\n",
    encoding="utf-8"
)

!git add README.md LICENSE requirements.txt
!git commit -m "docs: añadir README completo, licencia y requirements"

print("\nHistorial final:")
!git log --oneline
```

**Por qué estos archivos:**
- `README.md` es lo primero que ve cualquier persona en GitHub — es la portada del proyecto.
- `LICENSE` indica qué pueden hacer otros con tu código. Sin licencia, el código es técnicamente de uso restringido.
- `requirements.txt` es el estándar Python para declarar dependencias — `pip install -r requirements.txt` instala todo.
</details>

---
### Ejercicio 2 — Publicar en GitHub

**Enunciado:** Sube el repo `sistema_notas` a GitHub:

1. Crea el repositorio en GitHub (sin inicializarlo con README).
2. Añade el remote `origin` apuntando a tu repositorio.
3. Verifica con `git remote -v` que apunta al sitio correcto.
4. Haz el primer push con `-u` para establecer el tracking.
5. Verifica en el navegador que los archivos están en GitHub.
6. Muestra el estado del tracking con `git branch -vv`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Sustituye TU-USUARIO por tu nombre de usuario en GitHub

# !git remote add origin git@github.com:TU-USUARIO/sistema-notas.git
# !git remote -v
# !git branch -M main
# !git push -u origin main
# !git branch -vv


---
### Ejercicio 3 — El ciclo push/pull

**Enunciado:** Simula el flujo de trabajo diario:

1. Edita `gestor.py` localmente: añade un método `contar()` que devuelva el número de notas.
2. Commitea el cambio localmente.
3. Sube el cambio a GitHub con `git push`.
4. Verifica en el navegador que el cambio aparece en GitHub.
5. Ahora simula que estás en «otro ordenador»: clona tu propio repo en un directorio diferente (`sistema_notas_copia`).
6. Comprueba que `gestor.py` en la copia tiene el método `contar()`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import os, shutil
from pathlib import Path

# Asegurarnos de estar en sistema_notas
os.chdir(Path.home() / "sistema_notas")

# 1 y 2. Añadir método y commitear
gestor = Path("gestor.py").read_text(encoding="utf-8")
gestor += """
    def contar(self):
        return len(self._notas)
"""
Path("gestor.py").write_text(gestor, encoding="utf-8")

!git add gestor.py
!git commit -m "feat: añadir método contar en GestorNotas"

# 3. Subir a GitHub
# !git push

# 4. Verificar en el navegador (manual)
print("Verifica en: https://github.com/TU-USUARIO/sistema-notas")

# 5. Simular "otro ordenador" clonando en un directorio diferente
COPIA = Path.home() / "sistema_notas_copia"
if COPIA.exists():
    shutil.rmtree(COPIA)

os.chdir(Path.home())
# !git clone git@github.com:TU-USUARIO/sistema-notas.git sistema_notas_copia

# 6. Verificar que el método contar está en la copia
# os.chdir(COPIA)
# print(Path("gestor.py").read_text())

print("\nEl flujo completo:")
print("  Local: editar → add → commit → push")
print("  Otro ordenador: clone (o pull)")
```
</details>

---
### Ejercicio 4 — Explorar un repositorio open source

**Enunciado:** Sin hacer nada destructivo, explora un repositorio público real:

1. Clona `https://github.com/pallets/flask` (el framework web Flask) con `--depth 1` (shallow clone para que sea rápido).
2. Responde usando solo comandos Git:
   - ¿Cuántos archivos hay en el directorio raíz?
   - ¿Cuál es el mensaje del commit más reciente?
   - ¿Cuántos remotos tiene configurados?
   - ¿Qué ramas existen en el remoto?
3. Elimina el directorio clonado cuando termines.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import os, shutil
from pathlib import Path

FLASK = Path.home() / "flask_demo"
if FLASK.exists(): shutil.rmtree(FLASK)
os.chdir(Path.home())

# 1. Shallow clone (solo el último commit — mucho más rápido)
!git clone https://github.com/pallets/flask flask_demo --depth 1 -q
os.chdir(FLASK)

# 2a. Archivos en el directorio raíz
print("=== Archivos en la raíz ===")
!ls

# 2b. Último commit
print("\n=== Último commit ===")
!git log --oneline -1

# 2c. Remotos
print("\n=== Remotos ===")
!git remote -v

# 2d. Ramas remotas
print("\n=== Ramas remotas ===")
!git branch -r

# 3. Limpiar
os.chdir(Path.home())
shutil.rmtree(FLASK)
print("\nDirectorio eliminado")
```

**`--depth 1`** hace un *shallow clone* — descarga solo el último snapshot sin el historial completo. Para Flask (miles de commits) esto pasa de varios minutos a unos segundos. Útil cuando solo quieres leer el código, no contribuir.
</details>

---
### Ejercicio 5 (reto) — Script de publicación automática

**Enunciado:** Escribe una función Python `publicar_repo(ruta_local, nombre_github, usuario)` que automatice el proceso de publicar un repositorio local en GitHub:

1. Verifica que el directorio es un repo Git válido.
2. Verifica que no tiene ya un remote `origin`.
3. Genera y muestra las instrucciones paso a paso (no ejecuta nada destructivo — solo guía):
   - URL SSH del remote a añadir
   - Comandos a ejecutar en orden
   - URL del repo en GitHub donde verificar
4. Verifica el estado actual del repo (ramas, commits pendientes de push).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import subprocess
from pathlib import Path


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import subprocess
from pathlib import Path

def git_cmd(args, cwd):
    r = subprocess.run(["git"] + args, cwd=str(cwd),
                       capture_output=True, text=True)
    return r.stdout.strip(), r.returncode

def publicar_repo(ruta_local, nombre_github, usuario):
    ruta = Path(ruta_local).expanduser().resolve()

    print(f"\n{'═'*55}")
    print(f"  GUÍA DE PUBLICACIÓN — {ruta.name}")
    print(f"{'═'*55}")

    # 1. Verificar que es un repo Git
    _, code = git_cmd(["rev-parse", "--git-dir"], ruta)
    if code != 0:
        print(f"  ❌ '{ruta}' no es un repositorio Git")
        print(f"     Inicialízalo primero con: git init")
        return
    print(f"  ✅ Repositorio Git válido")

    # 2. Verificar si ya tiene remote origin
    remotos, _ = git_cmd(["remote"], ruta)
    if "origin" in remotos.splitlines():
        url, _ = git_cmd(["remote", "get-url", "origin"], ruta)
        print(f"  ⚠️  Ya tiene remote 'origin': {url}")
        print(f"     Elimínalo con: git remote remove origin")
        tiene_origin = True
    else:
        print(f"  ✅ Sin remote origin — listo para conectar")
        tiene_origin = False

    # 3. Estado del repo
    rama, _   = git_cmd(["branch", "--show-current"], ruta)
    commits, _ = git_cmd(["rev-list", "--count", "HEAD"], ruta)
    status, _ = git_cmd(["status", "--porcelain"], ruta)

    print(f"\n  ESTADO ACTUAL:")
    print(f"    Rama:        {rama or 'sin commits aún'}")
    print(f"    Commits:     {commits or '0'}")
    if status:
        print(f"    ⚠️  Tienes cambios sin commitear — haz commit antes de publicar")
    else:
        print(f"    ✅ Working tree limpio")

    # 4. Instrucciones
    ssh_url   = f"git@github.com:{usuario}/{nombre_github}.git"
    https_url = f"https://github.com/{usuario}/{nombre_github}"

    print(f"\n  PASOS PARA PUBLICAR:")
    print(f"    1. Crear el repo en GitHub (sin inicializar):")
    print(f"       https://github.com/new")
    print(f"       Nombre: {nombre_github}")
    print()
    if not tiene_origin:
        print(f"    2. Conectar el remoto:")
        print(f"       git remote add origin {ssh_url}")
        print()
    print(f"    3. Asegurar nombre de rama:")
    print(f"       git branch -M main")
    print()
    print(f"    4. Primer push:")
    print(f"       git push -u origin main")
    print()
    print(f"    5. Si tienes tags:")
    print(f"       git push --tags")
    print()
    print(f"  📍 Resultado: {https_url}")
    print(f"{'═'*55}")


publicar_repo("~/sistema_notas", "sistema-notas", "TU-USUARIO")
```
</details>

---
## Resumen de la clase

```bash
# ─── SSH ───────────────────────────────────────────────────
ssh-keygen -t ed25519 -C "email"   # generar clave
cat ~/.ssh/id_ed25519.pub           # mostrar clave pública
ssh -T git@github.com               # verificar conexión

# ─── Remotos ───────────────────────────────────────────────
git remote add origin <url>         # conectar con GitHub
git remote -v                       # ver remotos
git remote set-url origin <url>     # cambiar URL

# ─── Push ──────────────────────────────────────────────────
git push -u origin main             # primer push + tracking
git push                            # pushes siguientes
git push --tags                     # subir tags

# ─── Clone ─────────────────────────────────────────────────
git clone <url>                     # descargar repo completo
git clone <url> --depth 1           # shallow clone (rápido)

# ─── Pull / Fetch ──────────────────────────────────────────
git fetch origin                    # descargar sin fusionar
git pull                            # descargar y fusionar
git pull --rebase                   # descargar y rebase

# ─── Información ───────────────────────────────────────────
git branch -vv                      # ramas con info de tracking
git branch -r                       # ramas remotas
git log HEAD..origin/main           # commits pendientes de pull
```

## ¿Qué viene en la próxima clase?

En la **Clase 4** trabajamos con **ramas y colaboración** — la parte de Git que hace posible que varios desarrolladores trabajen en el mismo proyecto a la vez: `branch`, `switch`, `merge`, resolución de conflictos y el flujo de Pull Requests.